# COVID-19 Chest X-Ray Classification
### A Comparative Study of Dense, CNN, Deep CNN and LSTM Architectures

---

| | |
|---|---|
| **Task** | 3-class image classification of chest X-rays |
| **Classes** | `Covid` · `Normal` · `Viral Pneumonia` |
| **Data** | 251 training images · 66 test images |
| **Input** | 150 × 150 RGB, pixel values scaled to [0, 1] |
| **Framework** | TensorFlow / Keras |
| **Models compared** | Dense baseline · CNN · Deep CNN (Regularized) · Deep RNN (LSTM) |

---

## Table of Contents

1. [Introduction and Objectives](#1.-Introduction-and-Objectives)
2. [Data Pipeline and Preprocessing](#2.-Data-Pipeline-and-Preprocessing)
3. [Model 1: Dense Baseline](#3.-Model-1:-Dense-Baseline)
4. [Model 2: Convolutional Neural Network (CNN)](#4.-Model-2:-Convolutional-Neural-Network-(CNN))
5. [Model 3: Deep CNN (Regularized)](#5.-Model-3:-Deep-CNN-(Regularized))
6. [Model 4: Deep RNN (LSTM-based)](#6.-Model-4:-Deep-RNN-(LSTM-based))
7. [Conclusions](#7.-Conclusions)

## 1. Introduction and Objectives

Chest X-rays are one of the most widely available imaging tools for assessing lung
conditions. This notebook investigates whether a neural network can automatically
tell apart three categories of chest X-ray: **COVID-19**, **Viral Pneumonia**, and
**Normal**.

The dataset is small (251 training images), which makes this a useful test of how
model architecture choices behave when data is limited. Rather than tuning a single
model, four architectures of increasing complexity are trained under identical
conditions and compared.

**Objectives**

1. Build a reproducible data pipeline for the X-ray images.
2. Train four different architectures using the same optimizer, learning rate and
   stopping rule, so that differences in results come from the architecture itself.
3. Evaluate each model beyond overall accuracy, using per-class precision, recall,
   F1-score and confusion matrices.
4. Identify which architecture is best suited to this task and explain why.

**Common training setup (identical for all four models)**

| Setting | Value |
|---|---|
| Loss function | Categorical cross-entropy |
| Optimizer | Adam, learning rate 0.0001 |
| Maximum epochs | 30 |
| Early stopping | Monitor `val_accuracy`, patience 5, restore best weights |
| Batch size | 16 |

### Setup

Import the core libraries used throughout the notebook.

In [4]:
import tensorflow as tf
from tensorflow import keras
import os
import json
from pathlib import Path

## 2. Data Pipeline and Preprocessing

This section loads the images from disk, resizes and rescales them, and prepares
them for efficient training. The same pipeline feeds all four models, so every
model sees exactly the same data.

**What the cell below does**

| Step | Action | Why |
|---|---|---|
| 1.1 | Set paths, image size (150×150) and batch size (16) | Neural networks need identical input shapes |
| 1.2 | Load images from folders | Each subfolder name becomes the class label |
| 1.3 | Rescale pixels from 0–255 to 0–1 | Small, consistent inputs make training more stable |
| 1.4 | Apply `cache()` and `prefetch()` | Speeds up training; does not change results |
| 1.5 | Sanity check shapes and pixel range | Catches data problems before training starts |

In [2]:
# ------------------------------------------------------------------
# 1.1 Paths and settings
# ------------------------------------------------------------------
train_dir = "/Users/sanjib700/Desktop/My_Projects/Covide_image/Covid19-dataset/train"
test_dir  = "/Users/sanjib700/Desktop/My_Projects/Covide_image/Covid19-dataset/test"

IMG_SIZE = (150, 150)   # every image gets resized to this, regardless of its original size
BATCH_SIZE = 16         # number of images grouped together per training step

# ------------------------------------------------------------------
# 1.2 Load images directly from folders
#     Keras automatically uses each subfolder name as the class label
# ------------------------------------------------------------------
train_ds_raw = keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",   # one-hot labels, since we have 3 classes
    shuffle=True,
    seed=42,                    # fixed seed -> reproducible shuffle order
)

test_ds_raw = keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,               # no need to shuffle test data
)

class_names = train_ds_raw.class_names
num_classes = len(class_names)
print("Classes found:", class_names)

# ------------------------------------------------------------------
# 1.3 Preprocessing: rescale pixel values from 0-255 down to 0-1
#     Neural networks train more stably on small, consistent input ranges.
# ------------------------------------------------------------------
normalization_layer = keras.layers.Rescaling(1.0 / 255)

train_ds = train_ds_raw.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds_raw.map(lambda x, y: (normalization_layer(x), y))

# ------------------------------------------------------------------
# 1.4 Performance: cache + prefetch so training doesn't wait on disk I/O
#     cache()    -> keeps loaded/processed images in memory after the first epoch
#     prefetch() -> prepares the next batch while the current one is training
# ------------------------------------------------------------------
train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(tf.data.AUTOTUNE)

# ------------------------------------------------------------------
# 1.5 Sanity check: confirm shapes and pixel range before moving on
# ------------------------------------------------------------------
for images, labels in train_ds.take(1):
    print("\nOne batch of images shape:", images.shape)   # (batch, height, width, channels)
    print("One batch of labels shape:", labels.shape)     # (batch, num_classes)
    print("Pixel value range after scaling:", images.numpy().min(), "to", images.numpy().max())

Found 251 files belonging to 3 classes.


2026-09-24 04:07:53.479746: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-09-24 04:07:53.481014: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-09-24 04:07:53.481521: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-09-24 04:07:53.482181: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-09-24 04:07:53.483592: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Found 66 files belonging to 3 classes.
Classes found: ['Covid', 'Normal', 'Viral Pneumonia']

One batch of images shape: (16, 150, 150, 3)
One batch of labels shape: (16, 3)
Pixel value range after scaling: 0.0 to 1.0


2026-09-24 04:07:54.206115: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2026-09-24 04:07:54.233121: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### 2.1 Interpretation: Data Pipeline

- **Class labels come from the folder structure.** `image_dataset_from_directory`
  uses each subfolder name (`Covid`, `Normal`, `Viral Pneumonia`) as the label, so
  no manual label array is needed. The output confirms **251 training images** and
  **66 test images** across **3 classes**.
- **All images are resized to 150 × 150 pixels.** Source X-rays vary in resolution,
  but a neural network needs every input to share one shape.
- **Batches of 16 images** balance memory use against training speed and are a
  standard default for a dataset this size. The sanity check confirms the batch
  shape `(16, 150, 150, 3)` for images and `(16, 3)` for labels.
- **Labels are one-hot encoded** (`label_mode="categorical"`), for example
  Covid → `[1, 0, 0]`. This matches the `categorical_crossentropy` loss.
- **Training data is shuffled with a fixed seed (`seed=42`).** Shuffling stops the
  model from learning a fixed order (for example all Covid images first), and the
  seed keeps the run reproducible. Test data is not shuffled because order does not
  affect evaluation.
- **Pixels are rescaled from 0–255 to 0–1**, applied identically to train and test
  sets. The check confirms the range is now exactly 0.0 to 1.0.
- **`cache()` and `prefetch()` are speed optimizations only** and do not change
  model results.

> **Note on the test set:** the test split has only **66 images** (26 Covid, 20
> Normal, 20 Viral Pneumonia). Each single image is therefore worth about 1.5
> percentage points of accuracy, so small differences between models should be read
> with caution. This is revisited in the conclusions.

## 3. Model 1: Dense Baseline

The first model is a deliberately simple baseline. It flattens each image into a
single list of 67,500 numbers (150 × 150 × 3) and passes it through fully connected
layers. It has **no convolutional layers**, so it treats every pixel independently
and has no awareness of which pixels are neighbours.

A CNN or LSTM should only be considered worthwhile if it beats this baseline.

**Architecture:** `Flatten` → `BatchNormalization` → `Dense(300)` →
`BatchNormalization` → `Dense(100)` → `Dense(3, softmax)`
(about 20.6 million parameters)

`BatchNormalization` is included because an earlier version without it failed to
train (`loss: nan`), due to activations exploding when the very large flattened
input fed straight into a `Dense(300)` layer.

In [7]:
# ==============================================================================
# STEP 2: Dense Model — Build, Train, Evaluate
# ==============================================================================
# Requires Step 1 (data pipeline) to have already run: train_ds, test_ds,
# class_names, num_classes must all exist.

import numpy as np
import json
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix

# ------------------------------------------------------------------
# 2.1 Build the model
#     Flatten + Dense layers only -- no convolutional layers.
#     Baseline: treats every pixel independently, no spatial awareness.
# ------------------------------------------------------------------
dense_model = keras.Sequential([
    keras.layers.Input(shape=(150, 150, 3)),
    keras.layers.Flatten(),
    keras.layers.BatchNormalization(),          # stabilizes the huge flattened input
    keras.layers.Dense(300, activation="relu"),
    keras.layers.BatchNormalization(),          # stabilizes the huge flattened input
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(num_classes, activation="softmax"),
], name="Dense_Baseline")

dense_model.summary()

# ------------------------------------------------------------------
# 2.2 Compile
# ------------------------------------------------------------------
dense_model.compile(
    loss="categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),   # 10x smaller than default
    metrics=["accuracy"],
)

# ------------------------------------------------------------------
# 2.3 Train
#     EarlyStopping monitors validation accuracy, stops if it hasn't improved
#     in 5 epochs, and rolls back to the BEST epoch's weights (not the last one).
# ------------------------------------------------------------------
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
)

dense_history = dense_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=30,
    callbacks=[early_stop],
    verbose=2,
)

# Save the REAL training history to disk
Path("Training_history_jason").mkdir(exist_ok=True) # Create the folder
with open("Training_history_jason/dense_history.json", "w") as f:
    json.dump(dense_history.history, f)

# ------------------------------------------------------------------
# 2.4 Evaluate: overall test accuracy
# ------------------------------------------------------------------
test_loss, test_acc = dense_model.evaluate(test_ds, verbose=0)
print(f"\nDense model -- Test accuracy: {test_acc:.4f}")

# ------------------------------------------------------------------
# 2.5 Evaluate: full test set, per-class report + confusion matrix
#     A single accuracy number can hide a model doing well on one class
#     and poorly on another -- this checks every test image individually.
# ------------------------------------------------------------------
all_true = []
all_pred = []

for images, labels in test_ds:
    preds = dense_model.predict(images, verbose=0)
    all_true.extend(labels.numpy().argmax(axis=1))
    all_pred.extend(preds.argmax(axis=1))

all_true = np.array(all_true)
all_pred = np.array(all_pred)

print(f"\nTotal test images checked: {len(all_true)}")

print("\n=== Dense model -- per-class performance ===")
print(classification_report(all_true, all_pred, target_names=class_names))

print("=== Dense model -- confusion matrix ===")
print("Rows = true class, Columns = predicted class")
cm_dense = confusion_matrix(all_true, all_pred)
print("            ", "  ".join(f"{c[:8]:>8s}" for c in class_names))
for i, row in enumerate(cm_dense):
    print(f"{class_names[i][:12]:12s}", "  ".join(f"{v:8d}" for v in row))

Model: "Dense_Baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_2 (Flatten)             │ (None, 67500)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 67500)          │       270,000 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 300)            │    20,250,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 300)            │         1,200 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 3)              │           303 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,551,903 (78.40 MB)

 Trainable params: 20,416,303 (77.88 MB)

 Non-trainable params: 135,600 (529.69 KB)

Epoch 1/30
16/16 - 3s - 176ms/step - accuracy: 0.8446 - loss: 0.8497 - val_accuracy: 0.3939 - val_loss: 6.3833
Epoch 2/30
16/16 - 1s - 54ms/step - accuracy: 0.9482 - loss: 0.1136 - val_accuracy: 0.3939 - val_loss: 3.6598
Epoch 3/30
16/16 - 1s - 49ms/step - accuracy: 0.9841 - loss: 0.0592 - val_accuracy: 0.3939 - val_loss: 2.3289
Epoch 4/30
16/16 - 1s - 53ms/step - accuracy: 0.9960 - loss: 0.0390 - val_accuracy: 0.5152 - val_loss: 1.7051
Epoch 5/30
16/16 - 1s - 51ms/step - accuracy: 1.0000 - loss: 0.0295 - val_accuracy: 0.5455 - val_loss: 1.3089
Epoch 6/30
16/16 - 1s - 50ms/step - accuracy: 1.0000 - loss: 0.0237 - val_accuracy: 0.5909 - val_loss: 1.0723
Epoch 7/30
16/16 - 1s - 50ms/step - accuracy: 1.0000 - loss: 0.0200 - val_accuracy: 0.6515 - val_loss: 0.9251
Epoch 8/30
16/16 - 1s - 53ms/step - accuracy: 1.0000 - loss: 0.0171 - val_accuracy: 0.6818 - val_loss: 0.8226
Epoch 9/30
16/16 - 1s - 56ms/step - accuracy: 1.0000 - loss: 0.0150 - val_accuracy: 0.6970 - val_loss: 0.7479
Epoch 10/

2026-09-24 04:38:44.173731: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### 3.1 Interpretation: Dense Baseline

**Headline result: test accuracy 84.85%, macro F1 0.84.** This is a strong result
for a model with no spatial structure, and a large improvement over the first
attempt (without `BatchNormalization`), which failed to train at all.

**Training behaviour**

- **Training accuracy reached 100% by epoch 5** and stayed there while training loss
  kept shrinking (0.85 → 0.004). The model fits the 251 training images extremely
  tightly, which is expected for a very large Dense network on a small dataset.
- **Validation accuracy kept improving after training accuracy saturated**, rising
  from 39% to 85%. The model kept generalizing better over many epochs rather than
  only memorizing.
- **Validation loss began to creep up after epoch 17**, from its low of 0.563 to
  0.597 by epoch 27, while accuracy stayed flat. This is an early overfitting
  signal: predictions become less well calibrated even though the number of correct
  answers is unchanged.
- **Early stopping ended training at epoch 27.** Validation accuracy first reached
  its best value (0.8485) at epoch 22 and did not improve for 5 more epochs.
  `restore_best_weights=True` rolled the model back to the epoch-22 weights.

**Per-class performance**

| Class | Precision | Recall | F1 | Comment |
|---|---|---|---|---|
| Covid | 0.96 | 0.88 | 0.92 | Few false alarms; 3 of 26 cases missed |
| Normal | 0.83 | 0.75 | 0.79 | Weakest recall; 5 of 20 called Viral Pneumonia |
| Viral Pneumonia | 0.75 | 0.90 | 0.82 | Catches most cases but lowest precision |

**Confusion matrix.** The main confusion is between **Normal and Viral Pneumonia**
(5 Normal images predicted as Viral Pneumonia, 1 in the opposite direction). Covid
is rarely confused with either class, so it appears to be the most visually
distinct category in this dataset.

**Takeaway:** a solid baseline once training stability was fixed. The rising
validation loss suggests this architecture is near its limit on this amount of
data, which sets a clear reference point for the convolutional models.

### 3.2 Supplementary Check: Raw Predictions

The three short cells below inspect the Dense model's predictions directly. They
are kept as a quick verification that the evaluation loop lined up the true labels
and predictions correctly.

- `all_true` – the true class index of every test image (0 = Covid, 1 = Normal,
  2 = Viral Pneumonia). Because the test set is not shuffled, the labels appear in
  class order.
- `all_pred` – the Dense model's predicted class for each test image.
- The last cell prints the **softmax probability** for each class, one batch at a
  time. Values close to 1.0 mean a confident prediction; values split between two
  classes (for example `0.50 / 0.50`) show where the model was unsure.

In [33]:
print(all_true)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]


In [34]:
print(all_pred)

[0 0 0 0 0 0 0 0 0 1 2 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 2 1 1 1 2 2 2 1
 2 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 1 2 2 2 2 2 2 0 2 2 2 2]


In [28]:
for image, lables in test_ds:
    pred = dense_model.predict(image)
    print(pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
[[9.9640036e-01 3.4188062e-03 1.8081852e-04]
 [9.9999821e-01 4.3837272e-08 1.7382323e-06]
 [9.6748936e-01 2.2275511e-02 1.0235074e-02]
 [9.9699688e-01 6.6211132e-06 2.9965148e-03]
 [9.9996042e-01 7.0921385e-07 3.8807124e-05]
 [9.9552625e-01 1.3965842e-03 3.0771140e-03]
 [9.9966097e-01 3.1247223e-04 2.6532010e-05]
 [9.9865389e-01 1.2677021e-03 7.8352939e-05]
 [9.5707953e-01 4.1028652e-02 1.8917986e-03]
 [8.6527456e-05 9.8399377e-01 1.5919747e-02]
 [2.7743585e-02 2.0930730e-01 7.6294911e-01]
 [9.7218293e-01 1.8192774e-02 9.6241832e-03]
 [9.5731276e-01 3.9904330e-02 2.7829842e-03]
 [3.0938023e-01 6.8573606e-01 4.8836209e-03]
 [9.7621095e-01 9.7972772e-04 2.2809273e-02]
 [9.9966884e-01 1.1880578e-04 2.1236416e-04]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
[[9.99196589e-01 3.92691072e-06 7.99499685e-04]
 [9.98807311e-01 1.15968112e-03 3.29782088e-05]
 [9.99439657e-01 4.22261073e-04 1.38124422e-04]
 [9.99445379e-01 4.14838287e-04 1.39711148e-04]
 [9.9966096

2026-09-24 05:12:56.936506: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## 4. Model 2: Convolutional Neural Network (CNN)

The second model replaces the Flatten-and-Dense approach with **convolutional
layers**. These slide small filters across the image and detect local patterns such
as edges, textures and opacities, which are exactly the features a radiologist
looks at. Pooling layers then shrink the image while keeping the strongest signals.

**Architecture:** three `Conv2D` + `MaxPooling2D` blocks (16 → 32 → 64 filters) →
`Flatten` → `Dropout(0.5)` → `Dense(64)` → `Dense(3, softmax)`
(about 1.2 million parameters, roughly 17 times fewer than the Dense model)

`Dropout(0.5)` randomly switches off half the neurons during training, which
reduces overfitting on a small dataset.

In [38]:
# ==============================================================================
# STEP 3: CNN Model — Build, Train, Evaluate
# ==============================================================================
# Requires Step 1 (data pipeline) to have already run: train_ds, test_ds,
# class_names, num_classes must all exist.

import numpy as np
import json
from pathlib import Path
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix

# ------------------------------------------------------------------
# 3.1 Build the model
#     Conv2D + MaxPooling2D blocks -- exploits local spatial patterns
#     (edges, textures, opacities) that Dense layers ignore.
# ------------------------------------------------------------------
cnn_model = keras.Sequential([
    keras.layers.Input(shape=(150, 150, 3)),

    keras.layers.Conv2D(16, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D(2, 2),

    keras.layers.Conv2D(32, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D(2, 2),

    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D(2, 2),

    keras.layers.Flatten(),
    keras.layers.Dropout(0.5),          # randomly drops 50% of neurons during training,
                                          # reduces overfitting on a small (251-image) dataset
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(num_classes, activation="softmax"),
], name="CNN")

cnn_model.summary()

# ------------------------------------------------------------------
# 3.2 Compile
# ------------------------------------------------------------------
cnn_model.compile(
    loss="categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),   # matches Dense model's stable LR
    metrics=["accuracy"],
)

# ------------------------------------------------------------------
# 3.3 Train
#     EarlyStopping monitors validation accuracy, stops if it hasn't improved
#     in 5 epochs, and rolls back to the BEST epoch's weights (not the last one).
# ------------------------------------------------------------------
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
)

cnn_history = cnn_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=30,
    callbacks=[early_stop],
    verbose=2,
)

# Save the REAL training history to disk
Path("Training_history_jason").mkdir(exist_ok=True)   # Create the folder
with open("Training_history_jason/cnn_history.json", "w") as f:
    json.dump(cnn_history.history, f)

# ------------------------------------------------------------------
# 3.4 Evaluate: overall test accuracy
# ------------------------------------------------------------------
test_loss, test_acc = cnn_model.evaluate(test_ds, verbose=0)
print(f"\nCNN model -- Test accuracy: {test_acc:.4f}")

# ------------------------------------------------------------------
# 3.5 Evaluate: full test set, per-class report + confusion matrix
#     A single accuracy number can hide a model doing well on one class
#     and poorly on another -- this checks every test image individually.
# ------------------------------------------------------------------
all_true = []
all_pred = []

for images, labels in test_ds:
    preds = cnn_model.predict(images, verbose=0)
    all_true.extend(labels.numpy().argmax(axis=1))
    all_pred.extend(preds.argmax(axis=1))

all_true = np.array(all_true)
all_pred = np.array(all_pred)

print(f"\nTotal test images checked: {len(all_true)}")

print("\n=== CNN model -- per-class performance ===")
print(classification_report(all_true, all_pred, target_names=class_names))

print("=== CNN model -- confusion matrix ===")
print("Rows = true class, Columns = predicted class")
cm_cnn = confusion_matrix(all_true, all_pred)
print("            ", "  ".join(f"{c[:8]:>8s}" for c in class_names))
for i, row in enumerate(cm_cnn):
    print(f"{class_names[i][:12]:12s}", "  ".join(f"{v:8d}" for v in row))

Model: "CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 18496)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 18496)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │     1,183,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,207,587 (4.61 MB)

 Trainable params: 1,207,587 (4.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
16/16 - 2s - 106ms/step - accuracy: 0.5020 - loss: 1.0195 - val_accuracy: 0.6061 - val_loss: 0.9379
Epoch 2/30
16/16 - 0s - 28ms/step - accuracy: 0.6972 - loss: 0.7813 - val_accuracy: 0.6212 - val_loss: 0.7273
Epoch 3/30
16/16 - 0s - 27ms/step - accuracy: 0.8287 - loss: 0.5598 - val_accuracy: 0.7121 - val_loss: 0.5846
Epoch 4/30
16/16 - 1s - 32ms/step - accuracy: 0.8287 - loss: 0.4541 - val_accuracy: 0.7576 - val_loss: 0.4861
Epoch 5/30
16/16 - 0s - 29ms/step - accuracy: 0.8685 - loss: 0.3162 - val_accuracy: 0.7727 - val_loss: 0.4570
Epoch 6/30
16/16 - 0s - 29ms/step - accuracy: 0.8685 - loss: 0.2894 - val_accuracy: 0.8030 - val_loss: 0.4230
Epoch 7/30
16/16 - 0s - 31ms/step - accuracy: 0.9124 - loss: 0.2427 - val_accuracy: 0.8182 - val_loss: 0.3936
Epoch 8/30
16/16 - 0s - 29ms/step - accuracy: 0.9163 - loss: 0.1998 - val_accuracy: 0.8333 - val_loss: 0.3743
Epoch 9/30
16/16 - 0s - 29ms/step - accuracy: 0.9203 - loss: 0.1893 - val_accuracy: 0.8485 - val_loss: 0.3725
Epoch 10/

2026-09-24 05:37:00.774568: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### 4.1 Interpretation: CNN

**Headline result: test accuracy 90.91%, macro F1 0.90.** This is the best result of
the four models and clearly ahead of the Dense baseline (84.85%), despite the CNN
having far fewer parameters.

**Training behaviour**

- **Training was stable from the first epoch**, with no `nan` losses and no
  collapse. Training and validation accuracy climbed together, and validation loss
  fell smoothly from 0.938 to 0.263 across nearly the whole run.
- **Training accuracy never saturated at 100%** (peak about 98.4%), unlike the
  Dense model. Together with the smooth validation curve, this suggests the CNN is
  learning general patterns rather than memorizing the training set.
- **Early stopping ended training at epoch 20.** Validation accuracy peaked at
  0.9091 in epoch 15 and did not improve afterwards, so the epoch-15 weights were
  restored.

**Per-class performance**

| Class | Precision | Recall | F1 | Comment |
|---|---|---|---|---|
| Covid | 0.96 | 0.96 | 0.96 | Excellent and symmetric; 1 of 26 missed |
| Normal | 0.83 | 0.95 | 0.88 | Nearly all found; some Viral Pneumonia pulled in |
| Viral Pneumonia | 0.94 | 0.80 | 0.86 | Weakest recall; 4 of 20 missed |

**Confusion matrix.** Only **6 of 66** images are misclassified, compared with 10
for the Dense model. The remaining errors again concentrate at the **Normal /
Viral Pneumonia** boundary (3 Viral Pneumonia cases predicted as Normal), the same
pattern seen in the baseline, but at lower frequency.

**Takeaway:** the CNN outperforms the Dense baseline on accuracy and macro F1, and
trains in a more stable, better-behaved way. This supports the expectation that
convolutional layers suit image data because they exploit local spatial structure.

## 5. Model 3: Deep CNN (Regularized)

This model tests a common assumption: that **more depth and stronger regularization
will improve results**. It extends the CNN to four convolutional blocks, adds
`BatchNormalization` after each, and uses two Dropout layers.

**Architecture:** four `Conv2D` + `BatchNormalization` + `MaxPooling2D` blocks
(32 → 64 → 128 → 128 filters) → `Flatten` → `Dropout(0.5)` → `Dense(128)` →
`Dropout(0.3)` → `Dense(3, softmax)` (about 1.57 million parameters)

The question being asked is whether this extra capacity helps or hurts when only
251 training images are available.

In [39]:
# ==============================================================================
# STEP 4: Deep CNN (Regularized) Model — Build, Train, Evaluate
# ==============================================================================
# Requires Step 1 (data pipeline) to have already run: train_ds, test_ds,
# class_names, num_classes must all exist.

import numpy as np
import json
from pathlib import Path
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix

# ------------------------------------------------------------------
# 4.1 Build the model
#     4x Conv2D + BatchNormalization + MaxPooling2D blocks, with heavier
#     Dropout. Tests whether more depth + stronger regularization helps,
#     or whether it's too much for a 251-image training set.
# ------------------------------------------------------------------
deep_cnn_model = keras.Sequential([
    keras.layers.Input(shape=(150, 150, 3)),

    keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2, 2),

    keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2, 2),

    keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2, 2),

    keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2, 2),

    keras.layers.Flatten(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(num_classes, activation="softmax"),
], name="Deep_CNN_Regularized")

deep_cnn_model.summary()

# ------------------------------------------------------------------
# 4.2 Compile
# ------------------------------------------------------------------
deep_cnn_model.compile(
    loss="categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),   # matches other models' stable LR
    metrics=["accuracy"],
)

# ------------------------------------------------------------------
# 4.3 Train
#     EarlyStopping monitors validation accuracy, stops if it hasn't improved
#     in 5 epochs, and rolls back to the BEST epoch's weights (not the last one).
# ------------------------------------------------------------------
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
)

deep_cnn_history = deep_cnn_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=30,
    callbacks=[early_stop],
    verbose=2,
)

# Save the REAL training history to disk
Path("Training_history_jason").mkdir(exist_ok=True)   # Create the folder
with open("Training_history_jason/deep_cnn_history.json", "w") as f:
    json.dump(deep_cnn_history.history, f)

# ------------------------------------------------------------------
# 4.4 Evaluate: overall test accuracy
# ------------------------------------------------------------------
test_loss, test_acc = deep_cnn_model.evaluate(test_ds, verbose=0)
print(f"\nDeep CNN model -- Test accuracy: {test_acc:.4f}")

# ------------------------------------------------------------------
# 4.5 Evaluate: full test set, per-class report + confusion matrix
#     A single accuracy number can hide a model doing well on one class
#     and poorly on another -- this checks every test image individually.
# ------------------------------------------------------------------
all_true = []
all_pred = []

for images, labels in test_ds:
    preds = deep_cnn_model.predict(images, verbose=0)
    all_true.extend(labels.numpy().argmax(axis=1))
    all_pred.extend(preds.argmax(axis=1))

all_true = np.array(all_true)
all_pred = np.array(all_pred)

print(f"\nTotal test images checked: {len(all_true)}")

print("\n=== Deep CNN model -- per-class performance ===")
print(classification_report(all_true, all_pred, target_names=class_names))

print("=== Deep CNN model -- confusion matrix ===")
print("Rows = true class, Columns = predicted class")
cm_deep_cnn = confusion_matrix(all_true, all_pred)
print("            ", "  ".join(f"{c[:8]:>8s}" for c in class_names))
for i, row in enumerate(cm_deep_cnn):
    print(f"{class_names[i][:12]:12s}", "  ".join(f"{v:8d}" for v in row))

Model: "Deep_CNN_Regularized"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 150, 150, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 150, 150, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 75, 75, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 75, 75, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 75, 75, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 37, 37, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 37, 37, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 37, 37, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 18, 18, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 18, 18, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 18, 18, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 9, 9, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 10368)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10368)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 128)            │     1,327,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,569,859 (5.99 MB)

 Trainable params: 1,569,155 (5.99 MB)

 Non-trainable params: 704 (2.75 KB)

Epoch 1/30
16/16 - 4s - 268ms/step - accuracy: 0.6892 - loss: 1.8536 - val_accuracy: 0.3030 - val_loss: 1.0681
Epoch 2/30
16/16 - 1s - 82ms/step - accuracy: 0.9044 - loss: 0.4495 - val_accuracy: 0.6061 - val_loss: 1.0837
Epoch 3/30
16/16 - 1s - 73ms/step - accuracy: 0.9124 - loss: 0.5175 - val_accuracy: 0.5455 - val_loss: 1.1623
Epoch 4/30
16/16 - 1s - 72ms/step - accuracy: 0.9044 - loss: 0.3499 - val_accuracy: 0.3939 - val_loss: 1.3815
Epoch 5/30
16/16 - 1s - 72ms/step - accuracy: 0.9602 - loss: 0.1455 - val_accuracy: 0.3939 - val_loss: 1.6228
Epoch 6/30
16/16 - 1s - 70ms/step - accuracy: 0.9641 - loss: 0.1127 - val_accuracy: 0.3939 - val_loss: 2.0255
Epoch 7/30
16/16 - 1s - 90ms/step - accuracy: 0.9641 - loss: 0.1738 - val_accuracy: 0.3939 - val_loss: 2.4916

Deep CNN model -- Test accuracy: 0.6061

Total test images checked: 66

=== Deep CNN model -- per-class performance ===
                 precision    recall  f1-score   support

          Covid       0.61      0.96      0.75    

2026-09-24 05:48:13.066570: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
/Applications/Anaconda/anaconda3/envs/tf_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Applications/Anaconda/anaconda3/envs/tf_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Applications/Anaconda/anaconda3/envs/tf_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning

### 5.1 Interpretation: Deep CNN (Regularized)

**Headline result: test accuracy 60.61%, macro F1 0.47.** This is well below both
the Dense baseline (84.85%) and the CNN (90.91%). The model does separate classes to
some extent, but fails completely on one of them.

**The Normal class was never predicted.** Precision, recall and F1 are all exactly
0.00 for Normal. All 20 true Normal images were misclassified (11 as Covid, 9 as
Viral Pneumonia). The model has effectively given up on one class while still
partly distinguishing the other two.

**Training behaviour**

- **Training accuracy looked healthy in isolation**, rising from 69% to 96% by
  epoch 5.
- **Validation loss tells the real story.** While training loss stayed low
  (0.11–0.17), validation loss rose almost every epoch: 1.07 → 1.08 → 1.16 → 1.38 →
  1.62 → 2.03 → 2.49. This is a textbook overfitting pattern, in which the model
  gets better on training data and steadily worse on unseen data.
- **Validation accuracy was unstable**, swinging between 30% and 61% and sitting at
  0.3939 for epochs 4 to 7.
- **Early stopping ended training at epoch 7.** The best validation accuracy was
  0.6061 at epoch 2, and those weights were restored, matching the final test
  accuracy of 0.6061.

**Per-class performance**

| Class | Precision | Recall | F1 | Comment |
|---|---|---|---|---|
| Covid | 0.61 | 0.96 | 0.75 | Over-predicted: 16 Normal/Viral cases wrongly called Covid |
| Normal | 0.00 | 0.00 | 0.00 | Complete failure |
| Viral Pneumonia | 0.60 | 0.75 | 0.67 | Moderate |

**Likely cause.** Four convolutional blocks, `BatchNormalization` in every block and
two Dropout layers add a lot of depth and regularization relative to only 251
training images. The most plausible explanation is that this pushes the model toward
a degenerate solution that sacrifices the hardest class. This is a hypothesis
supported by the training curves, not something these experiments prove.

**Takeaway:** the simpler 3-block CNN achieved 90.91% with balanced classes on the
same data. On a dataset this small, **more depth and more regularization is not
automatically better** and here it clearly hurt. The result is reported as it
occurred, because a failed experiment is an informative finding.

## 6. Model 4: Deep RNN (LSTM-based)

The fourth model tests a completely different idea: treating an image as a
**sequence**. Each 150 × 150 × 3 image is reshaped into 150 "rows", where each row
is a list of 450 numbers (150 pixels × 3 colour channels). The LSTM reads the rows
one after another as if they were timesteps.

**Architecture:** `Reshape(150, 450)` → `LSTM(128)` → `Dropout(0.3)` → `LSTM(64)` →
`Dropout(0.3)` → `Dense(128)` → `Dropout(0.3)` → `Dense(3, softmax)`
(about 355 thousand parameters, the smallest of the four)

An LSTM is designed for ordered data such as text or time series. Using it on
images tests how much of the task depends on 2D spatial structure that this
reformulation partly discards.

In [40]:
# ==============================================================================
# STEP 4b: Deep RNN (LSTM-based) Model — Build, Train, Evaluate
# ==============================================================================
# Requires Step 1 (data pipeline) to have already run: train_ds, test_ds,
# class_names, num_classes must all exist.

import numpy as np
import json
from pathlib import Path
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix

# ------------------------------------------------------------------
# 4b.1 Build the model
#     Each 150x150x3 image is reshaped into a sequence of 150 "rows",
#     each row flattened to 150*3=450 features -> treated as a timestep.
#     2x LSTM layers (stacked), with Dropout for regularization.
# ------------------------------------------------------------------
deep_rnn_model = keras.Sequential([
    keras.layers.Input(shape=(150, 150, 3)),
    keras.layers.Reshape((150, 150 * 3)),   # (timesteps, features)

    keras.layers.LSTM(128, return_sequences=True),
    keras.layers.Dropout(0.3),

    keras.layers.LSTM(64),
    keras.layers.Dropout(0.3),

    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(num_classes, activation="softmax"),
], name="Deep_RNN_LSTM")

deep_rnn_model.summary()

# ------------------------------------------------------------------
# 4b.2 Compile
# ------------------------------------------------------------------
deep_rnn_model.compile(
    loss="categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),   # matches other models' stable LR
    metrics=["accuracy"],
)

# ------------------------------------------------------------------
# 4b.3 Train
#     EarlyStopping monitors validation accuracy, stops if it hasn't improved
#     in 5 epochs, and rolls back to the BEST epoch's weights (not the last one).
# ------------------------------------------------------------------
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
)

deep_rnn_history = deep_rnn_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=30,
    callbacks=[early_stop],
    verbose=2,
)

# Save the REAL training history to disk
Path("Training_history_jason").mkdir(exist_ok=True)
with open("Training_history_jason/deep_rnn_history.json", "w") as f:
    json.dump(deep_rnn_history.history, f)

# ------------------------------------------------------------------
# 4b.4 Evaluate: overall test accuracy
# ------------------------------------------------------------------
test_loss, test_acc = deep_rnn_model.evaluate(test_ds, verbose=0)
print(f"\nDeep RNN model -- Test accuracy: {test_acc:.4f}")

# ------------------------------------------------------------------
# 4b.5 Evaluate: full test set, per-class report + confusion matrix
# ------------------------------------------------------------------
all_true = []
all_pred = []

for images, labels in test_ds:
    preds = deep_rnn_model.predict(images, verbose=0)
    all_true.extend(labels.numpy().argmax(axis=1))
    all_pred.extend(preds.argmax(axis=1))

all_true = np.array(all_true)
all_pred = np.array(all_pred)

print(f"\nTotal test images checked: {len(all_true)}")

print("\n=== Deep RNN model -- per-class performance ===")
print(classification_report(all_true, all_pred, target_names=class_names))

print("=== Deep RNN model -- confusion matrix ===")
print("Rows = true class, Columns = predicted class")
cm_deep_rnn = confusion_matrix(all_true, all_pred)
print("            ", "  ".join(f"{c[:8]:>8s}" for c in class_names))
for i, row in enumerate(cm_deep_rnn):
    print(f"{class_names[i][:12]:12s}", "  ".join(f"{v:8d}" for v in row))

Model: "Deep_RNN_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape (Reshape)               │ (None, 150, 450)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 150, 128)       │       296,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 150, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 354,563 (1.35 MB)

 Trainable params: 354,563 (1.35 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
16/16 - 3s - 186ms/step - accuracy: 0.4462 - loss: 1.0317 - val_accuracy: 0.5455 - val_loss: 0.9447
Epoch 2/30
16/16 - 1s - 43ms/step - accuracy: 0.6534 - loss: 0.8685 - val_accuracy: 0.5758 - val_loss: 0.8632
Epoch 3/30
16/16 - 1s - 40ms/step - accuracy: 0.6454 - loss: 0.7841 - val_accuracy: 0.5606 - val_loss: 0.8150
Epoch 4/30
16/16 - 1s - 40ms/step - accuracy: 0.6653 - loss: 0.7056 - val_accuracy: 0.6212 - val_loss: 0.7310
Epoch 5/30
16/16 - 1s - 42ms/step - accuracy: 0.7410 - loss: 0.6373 - val_accuracy: 0.6061 - val_loss: 0.7386
Epoch 6/30
16/16 - 1s - 40ms/step - accuracy: 0.7331 - loss: 0.5802 - val_accuracy: 0.6212 - val_loss: 0.7027
Epoch 7/30
16/16 - 1s - 38ms/step - accuracy: 0.7291 - loss: 0.5797 - val_accuracy: 0.6667 - val_loss: 0.6920
Epoch 8/30
16/16 - 1s - 38ms/step - accuracy: 0.7809 - loss: 0.5067 - val_accuracy: 0.6364 - val_loss: 0.7014
Epoch 9/30
16/16 - 1s - 37ms/step - accuracy: 0.7888 - loss: 0.4837 - val_accuracy: 0.6364 - val_loss: 0.6789
Epoch 10/

2026-09-24 05:51:40.852509: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### 6.1 Interpretation: Deep RNN (LSTM-based)

**Headline result: test accuracy 69.70%, macro F1 0.68.** This is below the Dense
baseline (84.85%) and the CNN (90.91%), but better than the Deep CNN (60.61%).
Unlike the Deep CNN, it does not collapse on any class: all three get non-zero
precision and recall, although performance is uneven.

**Training behaviour**

- **Training accuracy rose steadily and smoothly**, from 44.6% to 90.8% by epoch 16.
  This is a slower, gentler climb than the Deep CNN's jump to 96% by epoch 5, which
  is consistent with recurrent models needing more epochs to fit their data.
- **Validation loss shows mild overfitting, far less severe than the Deep CNN.**
  Training loss fell from 1.03 to 0.27, while validation loss reached a low of
  0.652 at epoch 11, then drifted back up and oscillated between roughly 0.69 and
  0.79. It plateaus and wobbles rather than degrading steadily.
- **Early stopping ended training at epoch 16.** Validation accuracy peaked at
  0.6970 in epoch 11 (and again in epoch 15), then failed to improve for 5 epochs.
  The best weights were restored, matching the final test accuracy of 0.6970.

**Per-class performance**

| Class | Precision | Recall | F1 | Comment |
|---|---|---|---|---|
| Covid | 0.82 | 0.88 | 0.85 | Predicted well |
| Normal | 0.59 | 0.50 | 0.54 | Weakest; 10 of 20 misclassified (2 Covid, 8 Viral Pneumonia) |
| Viral Pneumonia | 0.62 | 0.65 | 0.63 | Confused with Normal in both directions |

**Likely explanation.** Treating image rows as timesteps gives the LSTM some
positional structure but discards the 2D locality (nearby pixels in both height and
width) that convolutions exploit directly. This produces a moderately useful model,
clearly better than a collapsed one, but weaker than a CNN on the harder classes.
As with the Deep CNN, this is an interpretation of the results rather than a tested
claim.

**Takeaway:** matching the architecture to the structure of the data (spatial for
images) matters more than model sophistication when training data is limited.

## 7. Conclusions

### 7.1 Summary of Results

All four models were trained on the same 251 images and evaluated on the same 66
test images, using the same optimizer, learning rate and stopping rule.

| Model | Parameters | Epochs run | Best val. acc. (best epoch) | Test accuracy | Macro F1 | Errors (of 66) |
|---|---|---|---|---|---|---|
| Dense baseline | 20.55 M | 27 | 0.8485 (22) | **84.85%** | 0.84 | 10 |
| **CNN** | **1.21 M** | 20 | 0.9091 (15) | **90.91%** | **0.90** | **6** |
| Deep CNN (Regularized) | 1.57 M | 7 | 0.6061 (2) | **60.61%** | 0.47 | 26 |
| Deep RNN (LSTM) | 0.35 M | 16 | 0.6970 (11) | **69.70%** | 0.68 | 20 |

**Per-class recall (share of true cases the model correctly identified)**

| Model | Covid | Normal | Viral Pneumonia |
|---|---|---|---|
| Dense baseline | 0.88 | 0.75 | 0.90 |
| **CNN** | **0.96** | **0.95** | 0.80 |
| Deep CNN (Regularized) | 0.96 | 0.00 | 0.75 |
| Deep RNN (LSTM) | 0.88 | 0.50 | 0.65 |

### 7.2 Main Findings

1. **The simple 3-block CNN is the best model.** It reached 90.91% accuracy and a
   macro F1 of 0.90, made the fewest errors (6 of 66), and was the only model with
   recall of at least 0.80 on every class. It also trained smoothly and did not
   memorize the training set (peak training accuracy about 98%, not 100%).

2. **Architecture mattered more than size or depth.** The ranking by test accuracy
   was CNN > Dense > RNN > Deep CNN. The best model has 17 times fewer parameters
   than the Dense baseline, and the smallest model (the LSTM) still beat the
   deeper, regularized CNN. Parameter count did not predict performance.

3. **More depth and more regularization actively hurt.** The Deep CNN added a fourth
   block, `BatchNormalization` and a second Dropout layer, and it fell from 90.91%
   to 60.61%, with validation loss climbing steadily from 1.07 to 2.49 and the
   Normal class never predicted. On only 251 training images, extra capacity and
   stacked regularization were not a safe default.

4. **Spatial structure is what the task needs.** The Dense baseline ignores it
   completely, the LSTM only partly preserves it, and the CNN exploits it directly.
   Results follow that order, which supports the view that convolutional layers are
   the natural fit for chest X-ray images.

5. **Covid is the easiest class; Normal versus Viral Pneumonia is the hard
   boundary.** Covid recall was 0.88 or higher for every model, and 0.96 for the
   two CNNs. In three of the four models, most remaining errors were confusions
   between Normal and Viral Pneumonia. In the Deep CNN, Normal was wrongly assigned
   to Covid as well (11 of 20). This suggests Normal and Viral Pneumonia images
   share visual features that are harder to separate than either is from Covid.

6. **Overfitting appeared in different forms.** The Dense model saturated at 100%
   training accuracy with a mild late rise in validation loss. The Deep CNN showed
   severe, rapid divergence. The LSTM showed mild oscillation. The CNN showed the
   least. Monitoring validation loss alongside accuracy was essential, since
   accuracy alone hid the Dense model's early overfitting.

7. **Early stopping worked as intended in every run**, ending training between
   epochs 7 and 27 and restoring the best weights in each case.

### 7.3 Limitations

These results should be treated as an initial comparison, not a validated clinical
finding. Several limitations affect how far they can be trusted.

- **The test set doubles as the validation set.** Every model used `test_ds` for
  `validation_data`, so early stopping and `restore_best_weights` selected the best
  epoch using the same images later reported as "test accuracy". The reported
  scores are therefore **optimistically biased**, and a separate held-out set
  would be needed for an unbiased estimate.
- **The test set is very small.** With 66 images, one image equals about 1.5
  percentage points. The gap between the CNN (6 errors) and the Dense model
  (10 errors) is 4 images, which is not large enough to rule out chance.
- **Each model was trained once.** No repeated runs, different seeds or
  cross-validation were used, so run-to-run variance is unknown. The Deep CNN's
  collapse in particular may be partly due to a single unlucky initialization.
- **No data augmentation was used**, even though it is a standard remedy for small
  image datasets and could change the ranking, especially for the deeper models.
- **Hyperparameters were not tuned.** All models used the same learning rate and
  patience, which may favour some architectures over others.
- **Explanations are hypotheses.** Statements about why the Deep CNN and LSTM
  underperformed are consistent with the training curves but were not tested with
  ablation experiments.
- **Clinical use is out of scope.** The dataset's source, patient diversity and
  image acquisition conditions are not characterized here, and nothing in this
  notebook supports diagnostic use.

### 7.4 Recommendations for Future Work

1. **Create a proper three-way split** (train, validation, test) so that model
   selection and final evaluation use different images.
2. **Use k-fold cross-validation** and repeated runs with different seeds to
   measure variance and give the model comparison statistical footing.
3. **Add data augmentation** (small rotations, zooms, brightness shifts) and
   re-test the Deep CNN, since it may recover once overfitting is controlled.
4. **Try transfer learning** with a network pre-trained on large image collections
   (for example ResNet or EfficientNet). This is usually the strongest option when
   only a few hundred labelled images are available.
5. **Run ablations on the Deep CNN**, removing `BatchNormalization` and one Dropout
   layer at a time, to find out which change caused the collapse.
6. **Target the Normal / Viral Pneumonia boundary**, for example with class-weighted
   loss, more examples of these classes, or inspection of misclassified images with
   a tool such as Grad-CAM to see what the model is looking at.

### 7.5 Final Conclusion

On this dataset, a compact three-block CNN is the most accurate and most reliable of
the four architectures compared, reaching **90.91% test accuracy** with balanced
performance across all three classes. The comparison shows that on small medical
imaging datasets, **choosing an architecture that matches the structure of the data
matters more than adding depth, parameters or regularization**. The main weakness
shared by all models is separating Normal from Viral Pneumonia. Because the test set
was also used for model selection and is small, these figures should be confirmed
with a proper held-out split and repeated runs before being relied upon.